In [77]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import re
import datetime
from time import sleep
import random

In [93]:
#extract articles link
#day/month/year

article_links = list()
d = datetime.date(2008,1,day=1)
while (d.year < 2010):
    url = 'http://archive.indianexpress.com/archive/news/'+str(d.day)+'/'+str(d.month)+'/'+str(d.year)+'/'
    try:
        response = requests.get(url)
    except Exception as e:
        print(e)
        pass
    d = d + datetime.timedelta(days=1)
    
    soup = bs(response.content,'html.parser')
    for val in soup.find_all('a',href=True):
        if (len(re.findall('.*(/[0-9]*/)$',val['href']))>0) & ("target" not in val.attrs.keys()):
            article_links.append(val['href'])
            
    sleep(random.random()*random.randint(11,37))


#store all the article links in a csv file
source_articles = pd.DataFrame()
source_articles['link'] = article_links
source_articles.to_csv('source_articles.csv')

HTTPConnectionPool(host='archive.indianexpress.com', port=80): Max retries exceeded with url: /archive/news/7/10/2009/ (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1211abc18>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))
Time taken: 0:46:54.608303
180255


In [371]:
#extract the actual content from the article links - INDIAN EXPRESS

start_time = datetime.datetime.now()
#article_dict = dict()
#i=0
for article in article_links[40001:43001]:
    try:
        #get rid of multipage in for the content by adding a trailing 0
        article_one_page = requests.get(article+'0')
        content = re.findall('<!--Start BOTTOM Story Body -->(.*?)<!--End BOTTOM Story Body -->',str(soup),re.DOTALL)[0]
    except Exception as e:
        print("Problem with: {}".format(article))
        #print datetime to track 
        print("Problem is: {} at {}".format(e,datetime.datetime.now()))
        print("Number of articles retrieved: {}".format(len(article_dict)))
        pass
    
    soup = bs(article_one_page.content,'html.parser')
    #finding all divs and then searching for the content only in divs
    #c = '#####'.join([str(ele) for ele in soup.find_all('div')])
    '''directly convert all the content in soup to str and then find content, will not make much difference in 
    processing because almost everything in the page is in a div tag'''
    
    article_dict[soup.title.string] = content
    
    if len(article_dict.keys()) > 1000:
        content_df = pd.DataFrame()
        content_df['Heading'], content_df['Content'] = article_dict.keys(), article_dict.values()
        content_df.to_csv('/Users/gurumanikanta/Documents/DS-Analytics/SupervisedLearning/TPI/IndianExpress/'+str(i)+'.csv')
        i+=1
        article_dict.clear()
    
    
    sleep(random.random()*random.randint(2,17))

print('Time taken: {}'.format(datetime.datetime.now()-start_time))

Problem with: http://archive.indianexpress.com/news/embarrassing-but-defeat-in-practice-game-doe/384333/
Problem is: HTTPConnectionPool(host='archive.indianexpress.com', port=80): Max retries exceeded with url: /news/embarrassing-but-defeat-in-practice-game-doe/384333/0 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1442f7f60>: Failed to establish a new connection: [Errno 60] Operation timed out')) at 2019-11-13 14:54:29.755268
Number of articles retrieved: 688
Time taken: 5:52:41.729758


In [372]:
article_links.index(article)

43000

In [374]:
len(article_dict)

31